# 04 — Model Training & Evaluation

**Goal:** train a linear regression baseline and a random forest on the extracted battery features,
evaluate both on a held-out battery, and produce SHAP explanations.

**Data:** `data/processed/battery_features.csv` (output of notebook 02)

**Train/test split strategy — battery-level, not cycle-level**

This is the most important design decision in this notebook.
If we split randomly across all cycles, cycles from the same battery appear in both
train and test — the model can effectively interpolate between adjacent cycles it has already seen.
That inflates performance metrics and does not reflect real deployment.

Instead we hold out **one entire battery** (B0018) and train on the other three.
The model has never seen B0018's degradation trajectory during training.

| Split | Batteries | Cycles |
|-------|-----------|--------|
| Train | B0005, B0006, B0007 | ~504 |
| Test  | B0018 | ~132 |

## 1 — Imports & paths

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('../src').resolve()))

from models         import build_linear_regression, build_random_forest, save_model
from evaluation     import evaluate_regression, compare_models, plot_actual_vs_predicted, plot_residuals
from explainability import compute_shap, plot_shap_summary, plot_shap_bar, plot_shap_waterfall, top_drivers

PROCESSED = Path('../data/processed')
ARTIFACTS = Path('../outputs/model_artifacts')
FIGURES   = Path('../outputs/figures')
ARTIFACTS.mkdir(parents=True, exist_ok=True)

print('Imports OK')

## 2 — Load data & define features

In [ ]:
df = pd.read_csv(PROCESSED / 'battery_features.csv')
print(f'Loaded {len(df)} rows, {df["battery_id"].nunique()} batteries')
df.groupby('battery_id').size().rename('cycles')

In [ ]:
# Features for SOH prediction — purely from discharge time-series
# (voltage, temperature, time arrays). We deliberately exclude:
#   - 'capacity'           : SOH = capacity / initial_capacity → direct leakage
#   - 'capacity_fade_rate' : derived from capacity → indirect leakage
# This forces the model to infer degradation from physical signatures alone.
FEATURE_COLS = [
    'discharge_num',          # cycle counter — known in deployment
    'mean_discharge_voltage', # voltage under load drops as resistance rises
    'min_discharge_voltage',  # end-of-discharge voltage behaviour
    'mean_temperature',       # temperature signature of the discharge
    'max_temperature',        # peak thermal stress
    'discharge_duration',     # aged cell empties faster
]

SOH_TARGET = 'soh'

# Train = B0005, B0006, B0007  |  Test = B0018
train_df = df[df['battery_id'] != 'B0018'].copy()
test_df  = df[df['battery_id'] == 'B0018'].copy()

X_train = train_df[FEATURE_COLS]
y_train = train_df[SOH_TARGET]

X_test  = test_df[FEATURE_COLS]
y_test  = test_df[SOH_TARGET]

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Features: {FEATURE_COLS}')

## 3 — Baseline: Linear Regression

Linear regression is our sanity check. If it performs well, the relationships in the data are
largely linear and a simple model suffices. If the random forest is significantly better,
that tells us there is useful non-linearity or feature interaction the linear model misses.

The model is wrapped in a `StandardScaler` pipeline (see `src/models.py`).

In [ ]:
lr = build_linear_regression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
results_lr = evaluate_regression(y_test, y_pred_lr, model_name='LinearRegression', target_name='SOH')

# Show coefficients (on the scaled features)
coefs = pd.Series(
    lr.named_steps['model'].coef_,
    index=FEATURE_COLS
).sort_values(key=abs, ascending=False)
print('\nCoefficients (on scaled features):')
print(coefs.round(5).to_string())

In [ ]:
plot_actual_vs_predicted(
    y_test, y_pred_lr,
    model_name='Linear Regression',
    discharge_nums=test_df['discharge_num'].values,
    save_path=FIGURES / 'lr_soh_predictions.png',
)

## 4 — Main model: Random Forest

Random forests handle non-linear relationships and feature interactions automatically.
They are also robust to outliers and do not require feature scaling.

300 trees with `min_samples_leaf=2` — see `src/models.py` for hyperparameter rationale.

In [ ]:
rf = build_random_forest()
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
results_rf = evaluate_regression(y_test, y_pred_rf, model_name='RandomForest', target_name='SOH')

plot_actual_vs_predicted(
    y_test, y_pred_rf,
    model_name='Random Forest',
    discharge_nums=test_df['discharge_num'].values,
    save_path=FIGURES / 'rf_soh_predictions.png',
)

In [ ]:
plot_residuals(
    y_test, y_pred_rf,
    model_name='Random Forest',
    discharge_nums=test_df['discharge_num'].values,
    save_path=FIGURES / 'rf_residuals.png',
)

## 5 — Model comparison

In [ ]:
comparison = compare_models([results_lr, results_rf])

## 6 — SHAP analysis (Random Forest)

SHAP values decompose each individual prediction into additive feature contributions.
The sum of all SHAP values for a prediction equals the difference between
that prediction and the global mean prediction.

We use `TreeExplainer` which is exact (no approximation) and fast for forests.

In [ ]:
shap_values = compute_shap(rf, X_train, X_test)
print(f'SHAP values shape: {shap_values.values.shape}')
print(f'Base value (global mean prediction): {shap_values.base_values[0]:.4f}')

In [ ]:
plot_shap_bar(
    shap_values,
    title='Mean |SHAP| — Global Feature Importance for SOH',
    save_path=FIGURES / 'shap_bar_soh.png',
)

In [ ]:
plot_shap_summary(
    shap_values,
    title='SHAP Summary — Feature Impact on SOH Prediction',
    save_path=FIGURES / 'shap_summary_soh.png',
)

### Waterfall plot for a single prediction

Pick a cycle near the 80 % warning boundary — this is the most
operationally interesting prediction to explain.

In [ ]:
# Find the cycle closest to the 80 % warning threshold in the test set
nearest_warning_idx = (y_test.values - 0.80).abs().argmin() \
    if hasattr(y_test.values - 0.80, 'abs') \
    else int(np.argmin(np.abs(y_test.values - 0.80)))

print(f'Explaining cycle index {nearest_warning_idx} ')
print(f'  Actual SOH    : {y_test.values[nearest_warning_idx]:.4f}')
print(f'  Predicted SOH : {y_pred_rf[nearest_warning_idx]:.4f}')
print(f'  {top_drivers(shap_values, nearest_warning_idx)}')

plot_shap_waterfall(
    shap_values,
    idx=nearest_warning_idx,
    title=f'SHAP Waterfall — cycle near 80% warning threshold',
    save_path=FIGURES / 'shap_waterfall_warning.png',
)

## 7 — RUL prediction

We now predict remaining useful life — how many cycles until the 80 % warning threshold.

**Important design note:** for RUL prediction we drop `discharge_num` from the features.
Why? Because `rul_warning = warning_cycle - discharge_num` — so if the model sees
`discharge_num` it can almost trivially compute the target. That would be data leakage,
not learning. We want the model to infer degradation state from *physical measurements*.

In [ ]:
RUL_FEATURES = [f for f in FEATURE_COLS if f != 'discharge_num']
RUL_TARGET   = 'rul_warning'

# Drop rows where RUL target is NaN (battery never reached the threshold)
train_rul = train_df.dropna(subset=[RUL_TARGET])
test_rul  = test_df.dropna(subset=[RUL_TARGET])

X_train_rul = train_rul[RUL_FEATURES]
y_train_rul = train_rul[RUL_TARGET]
X_test_rul  = test_rul[RUL_FEATURES]
y_test_rul  = test_rul[RUL_TARGET]

print(f'RUL train: {X_train_rul.shape}  |  RUL test: {X_test_rul.shape}')

In [ ]:
rf_rul = build_random_forest()
rf_rul.fit(X_train_rul, y_train_rul)
y_pred_rul = rf_rul.predict(X_test_rul)

results_rul = evaluate_regression(
    y_test_rul, y_pred_rul,
    model_name='RandomForest', target_name='RUL_warning (cycles)'
)

plot_actual_vs_predicted(
    y_test_rul, y_pred_rul,
    model_name='Random Forest',
    target_name='RUL (cycles to 80% warning)',
    discharge_nums=test_rul['discharge_num'].values,
    save_path=FIGURES / 'rf_rul_predictions.png',
    use_percent_formatter=False,   # RUL is in cycles, not a 0-1 fraction
)

## 8 — Save model artifacts

In [ ]:
save_model(lr,     ARTIFACTS / 'lr_soh.pkl')
save_model(rf,     ARTIFACTS / 'rf_soh.pkl')
save_model(rf_rul, ARTIFACTS / 'rf_rul_warning.pkl')

# Also save feature column lists so the Streamlit app loads them consistently
import json
meta = {
    'soh_features': FEATURE_COLS,
    'rul_features': RUL_FEATURES,
    'soh_target':   SOH_TARGET,
    'rul_target':   RUL_TARGET,
}
(ARTIFACTS / 'feature_meta.json').write_text(json.dumps(meta, indent=2))
print('feature_meta.json saved.')

print('\nAll artifacts saved:')
for f in sorted(ARTIFACTS.iterdir()):
    print(f'  {f.name}')

## 9 — What to check before building the dashboard

| Check | What to look for |
|-------|------------------|
| RF vs LR comparison | RF should have lower MAE and RMSE — if not, the data is very linear and LR suffices |
| SOH prediction plots | Error should be small and random — no systematic drift near EOL |
| Residual plot | Residuals scattered around 0 — no funnel shape (heteroscedasticity) |
| SHAP bar chart | `mean_discharge_voltage` and `discharge_duration` should dominate — consistent with correlation analysis |
| SHAP summary | Red (high feature value) dots for voltage/duration on the right = physically correct |
| Waterfall plot | Adds up to the actual prediction — sanity check that SHAP values are consistent |
| RUL MAE | Expect ~5–20 cycles MAE on held-out battery — acceptable for a planning tool |
| Saved artifacts | `lr_soh.pkl`, `rf_soh.pkl`, `rf_rul_warning.pkl`, `feature_meta.json` all present |